<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [61]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

import io

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from functools import partial

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import nms, box_iou

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [2]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [3]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [5]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        # Добавляй сюда свои аугментации при необходимости!
        A.RandomResizedCrop(size=(640, 640), scale=(0.6, 1.0), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.3),
        A.CoarseDropout(
            max_holes=8,
            max_height=32,
            max_width=32,
            fill_value=0,
            p=0.2
        ),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3, min_area=25)
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format='coco',
        label_fields=['labels']
    )
)

/tmp/ipykernel_1687/1813524390.py:10: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=0.3),
/tmp/ipykernel_1687/1813524390.py:11: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Не забываем инициализировать наш датасет

In [6]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [33]:
import timm
import torch
import torch.nn as nn

class Backbone(nn.Module):
    def __init__(self, model_name, out_indices=(-1, -2, -3), unfreeze_last=0):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=out_indices
        )
        for param in self.backbone.parameters():
            param.requires_grad = False

        if unfreeze_last > 0:
          blocks = list(self.backbone.children())
          for block in blocks[-unfreeze_last:]:
                for param in block.parameters():
                    param.requires_grad = True


    def forward(self, x):
        return self.backbone(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [34]:
import torch.nn as nn
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        self.out_channels = out_channels

        self.lateral_convs = nn.ModuleList()
        for in_channels in in_channels_list:
            self.lateral_convs.append(nn.Conv2d(in_channels, out_channels, kernel_size=1))

        self.output_convs = nn.ModuleList()
        for _ in range(len(in_channels_list)):
            self.output_convs.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))


    def forward(self, features):
        laterals = []
        for i, feat in enumerate(features):
            laterals.append(self.lateral_convs[i](feat))

        outputs = []
        prev = None
        for i in range(len(laterals) - 1, -1, -1):
            if prev is not None:
                prev = F.interpolate(prev, size=laterals[i].shape[-2:], mode='nearest')
                curr = laterals[i] + prev
            else:
                curr = laterals[i]
            curr = self.output_convs[i](curr)
            outputs.append(curr)
            prev = curr

        return outputs[::-1]

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

Выбран 2 вариант

In [35]:
class Head(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.num_classes = num_classes + 1

        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.cls_head = nn.Conv2d(in_channels, num_anchors * self.num_classes, kernel_size=1)
        self.reg_head = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        cls_logits = self.cls_head(x)
        bbox_preds = self.reg_head(x)
        return cls_logits, bbox_preds

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [36]:
from torchvision.models.detection.anchor_utils import AnchorGenerator

class Detector(nn.Module):
    def __init__(self,
                 backbone_model_name="efficientnet_b0",
                 neck_in_channels_list=None,
                 neck_out_channels=256,
                 num_classes=4,
                 anchor_sizes=((32, 64, 128), (64, 128, 256), (128, 256, 512)),
                 anchor_ratios=((0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0)),
                 input_size=(640, 640),
                 unfreeze_last=0):
      super().__init__()
      self.num_classes = num_classes
      self.input_size = input_size

      self.backbone = Backbone(backbone_model_name, out_indices=(-1, -2, -3), unfreeze_last=unfreeze_last)

      # Получаем каналы с backbone если не переданы
      if neck_in_channels_list is None:
          neck_in_channels_list = self.backbone.backbone.feature_info.channels()[-3:]

      # Neck (FPN)
      self.neck = Neck(neck_in_channels_list, neck_out_channels)

      # Head (без confidence)
      num_anchors = len(anchor_sizes[0]) * len(anchor_ratios[0])
      self.head = Head(neck_out_channels, num_anchors, num_classes)

      # Генерация якорей для каждого уровня FPN
      anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios)

      # Расчет grid sizes для каждого уровня
      reductions = [8, 16, 32]
      grid_sizes = []
      for reduction in reductions:
          grid_sizes.append([input_size[0] // reduction, input_size[1] // reduction])

      # Генерация якорей для всех уровней
      all_anchors = anchor_generator.grid_anchors(grid_sizes, strides=[[r, r] for r in reductions])

      # Объединяем все якоря в один тензор
      self.anchors = torch.cat(all_anchors, dim=0)

      # Предварительный расчет центров и размеров якорей
      self.anchor_centers = (self.anchors[:, :2] + self.anchors[:, 2:]) / 2
      self.anchor_sizes = self.anchors[:, 2:] - self.anchors[:, :2]

    def forward(self, x):
        # Backbone
        features = self.backbone(x)

        # Neck
        neck_features = self.neck(features)

        # Head
        all_cls_logits = []
        all_bbox_preds = []

        for feat in neck_features:
            cls_logits, bbox_preds = self.head(feat)

            # Преобразование в формат [B, num_anchors * H * W, ...]
            N = x.shape[0]
            cls_logits = cls_logits.permute(0, 2, 3, 1).contiguous()
            cls_logits = cls_logits.view(N, -1, self.head.num_classes)

            bbox_preds = bbox_preds.permute(0, 2, 3, 1).contiguous()
            bbox_preds = bbox_preds.view(N, -1, 4)

            all_cls_logits.append(cls_logits)
            all_bbox_preds.append(bbox_preds)

        # Объединяем предсказания со всех уровней
        cls_logits = torch.cat(all_cls_logits, dim=1)
        bbox_preds = torch.cat(all_bbox_preds, dim=1)

        if self.training:
            return bbox_preds, cls_logits

        # Инференс: декодируем ббоксы
        bboxes = self.decode_bboxes(bbox_preds)
        return bboxes, cls_logits

    def decode_bboxes(self, bbox_offsets):
        tx = bbox_offsets[:, :, 0]
        ty = bbox_offsets[:, :, 1]
        tw = bbox_offsets[:, :, 2]
        th = bbox_offsets[:, :, 3]

        center_x = self.anchor_centers[:, 0] + torch.sigmoid(tx) * self.anchor_sizes[:, 0]
        center_y = self.anchor_centers[:, 1] + torch.sigmoid(ty) * self.anchor_sizes[:, 1]

        w = torch.exp(tw) * self.anchor_sizes[:, 0]
        h = torch.exp(th) * self.anchor_sizes[:, 1]

        x_min = center_x - w / 2
        y_min = center_y - h / 2

        return torch.stack([x_min, y_min, w, h], dim=-1)

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [37]:
def safe_logit(x):
    """ Безопасный расчет logit'ов. """
    eps = 1e-6
    x = torch.clamp(x, eps, 1 - eps)
    return torch.log(x / (1 - x))

def get_target_offset(anchor_box, gt_box):
    """ Расчитываем таргет как желаемые смещения от якорей до GT.

    anchor_box: torch.Tensor в формате (x_min, y_min, x_max, y_max),
    gt_box: torch.Tensor в формате (x_min, y_min, x_max, y_max).
    """
    # Конвертируем GT в формат (x_center, y_center), (w, h)
    gt_center = (gt_box[:2] + gt_box[2:]) / 2
    gt_size = gt_box[2:] - gt_box[:2]

    # Конвертируем якоря в формат (x_center, y_center), (w, h)
    anchor_center = (anchor_box[:2] + anchor_box[2:]) / 2
    anchor_size = anchor_box[2:] - anchor_box[:2]

    # Вычисляем значения смещений для положительных ббоксов
    tx = (gt_center[0] - anchor_center[0]) / anchor_size[0]
    ty = (gt_center[1] - anchor_center[1]) / anchor_size[1]
    target_tx = safe_logit(tx)
    target_ty = safe_logit(ty)

    target_tw = torch.log(gt_size[0] / anchor_size[0])
    target_th = torch.log(gt_size[1] / anchor_size[1])
    return torch.tensor([target_tx, target_ty, target_tw, target_th]).to(anchor_box.device)

In [15]:
from torchvision.ops import box_iou

In [38]:
def TAL_assigner(anchors, gt_boxes, gt_labels, num_classes, alpha=1.0, beta=6.0, topk=13):

    num_anchors = anchors.shape[0]
    target_offsets = torch.zeros((num_anchors, 4), device=anchors.device)
    target_cls = torch.zeros((num_anchors, num_classes + 1), device=anchors.device)
    target_cls[:, 0] = 1  # По умолчанию все якоря - фон

    if gt_boxes.numel() == 0:
        return target_offsets, target_cls

    # Конвертируем GT в формат (x_min, y_min, x_max, y_max)
    gt_xyxy = gt_boxes.clone()
    gt_xyxy[:, 2:] = gt_xyxy[:, :2] + gt_xyxy[:, 2:]

    # Вычисляем центры якорей
    anchor_centers = (anchors[:, :2] + anchors[:, 2:]) / 2

    # Вычисляем IoU между всеми якорями и всеми GT
    ious = box_iou(anchors, gt_xyxy)

    # Для каждого GT отбираем якоря, центры которых внутри GT
    # И вычисляем classification score s
    s = torch.zeros_like(ious)
    for gt_idx, gt_box in enumerate(gt_xyxy):
        # Проверяем, какие якоря внутри GT
        inside_mask = (anchor_centers[:, 0] >= gt_box[0]) & \
                      (anchor_centers[:, 0] <= gt_box[2]) & \
                      (anchor_centers[:, 1] >= gt_box[1]) & \
                      (anchor_centers[:, 1] <= gt_box[3])

        if inside_mask.sum() == 0:
            # Если нет якорей внутри, берем якоря с IoU > 0
            inside_mask = ious[:, gt_idx] > 0

        # Для якорей внутри GT, s = 1 (максимальная вероятность)
        s[inside_mask, gt_idx] = 1.0

    # Вычисляем метрику t = s^alpha * u^beta
    t = torch.pow(s, alpha) * torch.pow(ious, beta)  # shape (num_anchors, num_gt)

    # Для каждого GT выбираем top-k якорей с наибольшим t
    selected_anchors_mask = torch.zeros(num_anchors, dtype=torch.bool, device=anchors.device)
    best_gt_for_anchor = torch.zeros(num_anchors, dtype=torch.long, device=anchors.device)

    for gt_idx in range(len(gt_boxes)):
        t_for_gt = t[:, gt_idx]

        # Выбираем top-k
        topk_values, topk_indices = torch.topk(t_for_gt, min(topk, num_anchors))

        # Отмечаем выбранные якоря
        for idx in topk_indices:
            if t_for_gt[idx] > 0:
                selected_anchors_mask[idx] = True
                best_gt_for_anchor[idx] = gt_idx

    # Если предсказание подходит для нескольких GT, выбираем GT с наибольшим IoU
    for anchor_idx in range(num_anchors):
        if selected_anchors_mask[anchor_idx]:
            gt_idx = best_gt_for_anchor[anchor_idx]
            # Проверяем, нет ли другого GT с большим IoU
            ious_for_anchor = ious[anchor_idx]
            best_iou_gt = torch.argmax(ious_for_anchor)
            if ious_for_anchor[best_iou_gt] > ious_for_anchor[gt_idx]:
                gt_idx = best_iou_gt
            best_gt_for_anchor[anchor_idx] = gt_idx

    # Заполняем target_offsets и target_cls для выбранных якорей
    for anchor_idx in range(num_anchors):
        if selected_anchors_mask[anchor_idx]:
            gt_idx = best_gt_for_anchor[anchor_idx]

            # Вычисляем смещения
            target_offsets[anchor_idx] = get_target_offset(anchors[anchor_idx], gt_xyxy[gt_idx])

            # Заполняем target_cls (обнуляем фон и ставим 1 для нужного класса)
            target_cls[anchor_idx] = 0
            target_cls[anchor_idx, gt_labels[gt_idx] + 1] = 1

    return target_offsets, target_cls

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [17]:
from torchvision.ops import distance_box_iou_loss

In [39]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [40]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [41]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0103336572647095


In [42]:
def diou_loss(pred_boxes, gt_boxes):
    # Вычисляем площади обоих ббоксов
    # pred_boxes: (x1, y1, x2, y2)
    pred_x1, pred_y1, pred_x2, pred_y2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    gt_x1, gt_y1, gt_x2, gt_y2 = gt_boxes[:, 0], gt_boxes[:, 1], gt_boxes[:, 2], gt_boxes[:, 3]

    # Площади
    pred_area = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
    gt_area = (gt_x2 - gt_x1) * (gt_y2 - gt_y1)

    # Вычисляем пересечение
    # Координаты пересечения
    inter_x1 = torch.max(pred_x1, gt_x1)
    inter_y1 = torch.max(pred_y1, gt_y1)
    inter_x2 = torch.min(pred_x2, gt_x2)
    inter_y2 = torch.min(pred_y2, gt_y2)

    # Ширина и высота пересечения
    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter_area = inter_w * inter_h

    # Вычисляем объединение
    union_area = pred_area + gt_area - inter_area
    union_area = union_area.clamp(min=1e-6)

    # Вычисляем IoU
    iou = inter_area / union_area

    # Вычисляем центры ббоксов
    pred_center_x = (pred_x1 + pred_x2) / 2
    pred_center_y = (pred_y1 + pred_y2) / 2
    gt_center_x = (gt_x1 + gt_x2) / 2
    gt_center_y = (gt_y1 + gt_y2) / 2

    # Вычисляем квадрат расстояния между центрами
    d2 = (pred_center_x - gt_center_x) ** 2 + (pred_center_y - gt_center_y) ** 2

    # Вычисляем диагональ выпуклой оболочки
    # Координаты выпуклой оболочки (минимальный прямоугольник, содержащий оба ббокса)
    convex_x1 = torch.min(pred_x1, gt_x1)
    convex_y1 = torch.min(pred_y1, gt_y1)
    convex_x2 = torch.max(pred_x2, gt_x2)
    convex_y2 = torch.max(pred_y2, gt_y2)

    # Квадрат диагонали выпуклой оболочки
    c2 = (convex_x2 - convex_x1) ** 2 + (convex_y2 - convex_y1) ** 2
    c2 = c2.clamp(min=1e-6)

    # Вычисляем DIoU
    diou = 1 - iou + d2 / c2

    # Возвращаем усредненное значение
    return diou.mean()

In [43]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [ ]:

# Место для обучения, валидации и экспериментов!


Унификация формата ббоксов в HaloDataset

In [55]:
import io
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset

class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        boxes_xyxy = []
        for box in boxes:
            x1, y1, w, h = box
            boxes_xyxy.append([x1, y1, x1 + w, y1 + h])
        boxes = boxes_xyxy

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

Улучшенный детектор с предсказанием

In [63]:
class Backbone(nn.Module):
    def __init__(self, model_name="efficientnet_b0", out_indices=(-1,), unfreeze_last=0):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, features_only=True, out_indices=out_indices)

        # Замораживаем все слои
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Размораживаем последние k блоков
        if unfreeze_last > 0:
            blocks = list(self.backbone.children())
            for block in blocks[-unfreeze_last:]:
                for param in block.parameters():
                    param.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return features[0]  # возвращаем один feature map

In [64]:
class Neck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

In [66]:
class Head(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.num_classes = num_classes + 1  # +1 для фона

        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.cls_head = nn.Conv2d(in_channels, num_anchors * self.num_classes, kernel_size=1)
        self.reg_head = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        cls_logits = self.cls_head(x)
        bbox_preds = self.reg_head(x)
        return cls_logits, bbox_preds

In [70]:
class Detector(nn.Module):
    def __init__(self,
                 backbone_model_name="efficientnet_b0",
                 neck_out_channels=256,
                 num_classes=4,
                 anchor_sizes=((32, 64, 128),),
                 anchor_ratios=((0.5, 1.0, 2.0),),
                 input_size=(640, 640),
                 unfreeze_last=2):
        super().__init__()
        self.num_classes = num_classes
        self.input_size = input_size

        # Backbone (один выход)
        self.backbone = Backbone(backbone_model_name, out_indices=(-1,), unfreeze_last=unfreeze_last)

        # Получаем количество каналов
        in_channels = self.backbone.backbone.feature_info.channels()[0]

        # Neck
        self.neck = Neck(in_channels, neck_out_channels)

        # Head (без confidence)
        num_anchors = len(anchor_sizes[0]) * len(anchor_ratios[0])
        self.head = Head(neck_out_channels, num_anchors, num_classes)

        # Генерация якорей
        anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios)
        reduction = 32
        grid_sizes = [[input_size[0] // reduction, input_size[1] // reduction]]
        all_anchors = anchor_generator.grid_anchors(grid_sizes, strides=[[reduction, reduction]])

        # Регистрируем как buffer (автоматически будет на том же устройстве, что и модель)
        self.register_buffer("anchors", torch.cat(all_anchors, dim=0))

    def forward(self, x):
        features = self.backbone(x)
        neck_features = self.neck(features)

        cls_logits, bbox_preds = self.head(neck_features)

        N = x.shape[0]
        cls_logits = cls_logits.permute(0, 2, 3, 1).contiguous()
        cls_logits = cls_logits.view(N, -1, self.head.num_classes)

        bbox_preds = bbox_preds.permute(0, 2, 3, 1).contiguous()
        bbox_preds = bbox_preds.view(N, -1, 4)

        if self.training:
            return bbox_preds, cls_logits

        return bbox_preds, cls_logits

Обновленный ComputeLoss с DIoU

In [71]:
class ComputeLoss:
    def __init__(self, bbox_loss=None, cls_loss=None, weight_bbox=5, weight_cls=1):
        self.bbox_loss = bbox_loss if bbox_loss is not None else diou_loss
        self.cls_loss = nn.CrossEntropyLoss() if cls_loss is None else cls_loss
        self.weight_bbox = weight_bbox
        self.weight_cls = weight_cls

    def __call__(self, predicts, targets):
        pred_bboxes, pred_cls_logits = predicts
        target_boxes, target_cls = targets

        pred_cls_logits = pred_cls_logits.reshape(-1, pred_cls_logits.shape[-1])
        target_cls = target_cls.argmax(dim=1)

        loss_cls = self.cls_loss(pred_cls_logits, target_cls)

        pos_mask = target_cls != 0
        if pos_mask.sum() > 0:
            loss_bbox = self.bbox_loss(pred_bboxes[pos_mask], target_boxes[pos_mask])
        else:
            loss_bbox = torch.tensor(0.0, device=pred_bboxes.device)

        return self.weight_bbox * loss_bbox + self.weight_cls * loss_cls

Эксперименты: расширенные аугментации, обучение с разными assigner

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [73]:
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision.ops import nms

class Runner:
    """ Базовый класс для обучения и валидации модели.

    Параметры
    ---------
    model : torch модель, которая будет обучаться.
    compute_loss : экземпляр класса ComputeLoss (или другого с реализованным методом __call__).
    optimizer : torch optimizer
    train_dataloader : torch dataloader семплирующий данные для обучения модели.
    assign_target_method : callable, который решает задачу сопоставления якорей и таргета (например, assign_target)
    deivce : девайс на котором будет происходить обучения, по дефолту "cpu"
    scheduler : torch scheduler
    assign_target_kwargs : доп параметры для функции в `assign_target_method`,
    val_dataloader : torch dataloader загружающий валидационные данные.
    score_threshold : При расчете метрики на валидации, все предсказания,
        с (confidence score * cls_probs) < score_threshold будут проигнорированны.
    nms_threshold : Предсказания, имеющие пересечение по IoU >= nms_threshold будут считаться одним предсказанием.
    max_boxes_per_cls : Максимальное количество ббоксов на изображение для одного класса после фильтрации по `score_threshold`.
    """
    def __init__(self, model, compute_loss, optimizer, train_dataloader, assign_target_method, device=None,
                 scheduler=None, assign_target_kwargs=None,
                 val_dataloader=None, val_every=5, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8):
        self.model = model
        self.compute_loss = compute_loss
        self.optimizer = optimizer
        self.train_dataloader = train_dataloader
        assign_target_kwargs = {} if assign_target_kwargs is None else assign_target_kwargs
        self.assign_target_method = partial(assign_target_method, **assign_target_kwargs)
        self.device = "cpu" if device is None else device
        self.scheduler = scheduler

        # Валидационные параметры
        self.val_dataloader = val_dataloader
        self.val_every = val_every
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.max_boxes_per_cls = max_boxes_per_cls

        # Вспомогательные массивы
        self.batch_loss = []
        self.epoch_loss = []
        self.val_metric = []

    def _run_train_epoch(self, dataloader, verbose=True):
        """ Обучить модель одну эпоху на данных из `dataloader` """
        self.model.train()
        batch_loss = []
        for images, targets in (pbar := tqdm(dataloader, desc=f"Process train epoch", leave=False)):
            images = images.to(self.device)
            outputs = self.model(images)

            anchors = self.model.anchors.view(-1, 4)
            accum_loss = 0.0
            for ix in range(images.shape[0]):
                gt_boxes = targets[ix]['boxes'].to(self.device)
                gt_labels = targets[ix]['labels'].to(self.device)
                # выбираем какие якоря будут использоваться при расчете лосса.
                assigned_targets = self.assign_target_method(anchors, gt_boxes, gt_labels,
                                                             num_classes=model.num_classes)
                # Считаем лосс на основании предсказаний модели и таргетов.
                outputs_ixs = [out[ix] for out in outputs]
                loss = self.compute_loss(outputs_ixs, assigned_targets)
                accum_loss += loss
            accum_loss = accum_loss / images.shape[0]
            batch_loss.append(accum_loss.cpu().detach().item())

            # Делаем шаг оптимизатора после расчета лосса для всех элементов батча
            self.optimizer.zero_grad()
            accum_loss.backward()
            self.optimizer.step()
        # Обновляем описание tqdm бара усредненным значением лосса за предыдущй батч
            if verbose:
                pbar.set_description(f"Current batch loss: {batch_loss[-1]:.4}")
        return batch_loss

    def train(self, num_epochs=10, verbose=True):
        """ Обучаем модель заданное количество эпох. """
        val_desc = ""
        for epoch in (epoch_pbar := tqdm(range(1, num_epochs+1), desc="Train epoch", total=num_epochs)):
            # Обучаем модель одну эпоху
            loss = self._run_train_epoch(self.train_dataloader, verbose=verbose)
            self.batch_loss.extend(loss)
            self.epoch_loss.append(np.mean(self.batch_loss[-len(self.train_dataloader):]))

            # Делаем валидацию, если был передан валидационный датасет
            if self.val_dataloader is not None and epoch % self.val_every == 0:
                val_metric = self.validate()
                self.val_metric.append(val_metric)
                val_desc = f" Val {val_metric:.4}"

            # Обновляем описание tqdm бара усредненным значением лосса за предыдую эпоху
            if verbose:
                epoch_pbar.set_description(f"Last epoch loss: Train {self.epoch_loss[-1]:.4}" + val_desc)
            # Делаем шаг scheduler'a если он был передан
            if self.scheduler is not None:
                self.scheduler.step()

    @torch.no_grad()
    def validate(self, dataloader=None):
        """ Метод для валидации модели. Если dataloader не передан, будет использоваться self.val_dataloder.
        Возвращает mAP (0.5 ... 0.95).
        """
        self.model.eval()
        dataloader = self.val_dataloader if dataloader is None else dataloader
        # Считаем метрику mAP с помощью функции из torchmetrics
        metric = MeanAveragePrecision(box_format="xywh", iou_type="bbox")
        for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
            images = images.to(self.device)
            outputs = self.model(images)
            predicts = _filter_predictions(outputs, self.score_threshold, self.nms_threshold,
                                           max_boxes_per_cls=self.max_boxes_per_cls, return_type="torch")
            metric.update(predicts, targets)
        return metric.compute()["map"].item()

    def plot_loss(self, row_figsize=3):
        nrows = 2 if self.val_metric else 1
        _, ax = plt.subplots(nrows, 1, figsize=(12, row_figsize*nrows), tight_layout=True)
        ax = np.array([ax]) if not isinstance(ax, np.ndarray) else ax
        ax[0].plot(self.batch_loss, label="Train batch Loss", color="tab:blue")
        ax[0].plot(np.arange(1, len(self.batch_loss)+1, len(self.train_dataloader)), self.epoch_loss,
                   color="tab:orange", label="Train epoch Loss")
        ax[0].grid()
        ax[0].set_title("Train Loss")
        ax[0].set_xlabel("Number of Iterations")
        ax[0].set_ylabel("Loss")
        if self.val_metric:
            ax[1].plot(np.arange(self.val_every, len(self.batch_loss)+1, len(self.val_dataloader) * self.val_every),
                       np.array(self.val_metric) * 100, color="tab:green", label="Validation mAP")
            ax[1].grid()
            ax[1].set_title("Valiation mAP")
            ax[1].set_xlabel("Number of Iterations")
            ax[1].set_ylabel("mAP (%)")
        plt.legend()
        plt.show()

def _filter_predictions(predictions, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8, return_type="list"):
    """
    Фильтрация предсказаний для модели БЕЗ confidence score.
    predictions: (bboxes, cls_logits)
    """
    bboxes, cls_logits = predictions
    cls_probs = torch.softmax(cls_logits, dim=-1)

    num_classes = cls_probs.shape[-1]
    final_predictions = []

    for boxes, cls_probs_img in zip(bboxes, cls_probs):
        preds = {"boxes": [], "labels": [], "scores": []}

        # Пропускаем класс фона (0)
        for cls in range(1, num_classes):
            cls_scores = cls_probs_img[:, cls]
            keep_ixs = cls_scores > score_threshold
            if keep_ixs.sum() == 0:
                continue
            cls_boxes = boxes[keep_ixs]
            cls_scores = cls_scores[keep_ixs]

            if len(cls_boxes) > max_boxes_per_cls:
                pos = torch.argsort(cls_scores, descending=True)
                cls_boxes = cls_boxes[pos[:max_boxes_per_cls]]
                cls_scores = cls_scores[pos[:max_boxes_per_cls]]

            # Конвертируем в xyxy для NMS
            boxes_xyxy = cls_boxes.clone()
            boxes_xyxy[:, 2:] = boxes_xyxy[:, :2] + boxes_xyxy[:, 2:]
            pred_ixs = nms(boxes_xyxy, cls_scores, nms_threshold)

            for ix in pred_ixs:
                preds["boxes"].append(cls_boxes[ix].cpu().tolist())
                preds["labels"].append(cls - 1)
                preds["scores"].append(cls_scores[ix].item())

        if return_type == "torch":
            for key, item in preds.items():
                preds[key] = torch.tensor(item)
        final_predictions.append(preds)
    return final_predictions

функция assign_target

In [74]:
def assign_target(anchors, gt_boxes, gt_labels, num_classes, pos_th=0.6, neg_th=0.3):
    num_anchors = anchors.shape[0]
    target_boxes = torch.zeros((num_anchors, 4), device=anchors.device)
    target_cls = torch.zeros((num_anchors, num_classes + 1), device=anchors.device)
    target_cls[:, 0] = 1

    if gt_boxes.numel() == 0:
        return target_boxes, target_cls

    ious = box_iou(anchors, gt_boxes)
    best_iou, best_gt_idx = ious.max(dim=1)

    pos_mask = best_iou >= pos_th
    pos_indices = pos_mask.nonzero(as_tuple=True)[0]

    for pos in pos_indices:
        gt_idx = best_gt_idx[pos]
        target_boxes[pos] = gt_boxes[gt_idx]
        target_cls[pos] = 0
        target_cls[pos, gt_labels[gt_idx] + 1] = 1

    return target_boxes, target_cls

Создаем, обучаем и сравниваем модели с разными раннерами

In [78]:
# Очистка памяти
torch.cuda.empty_cache()

In [80]:
from functools import partial
from torch.utils.data import DataLoader
import torch.optim as optim

train_transform = A.Compose([
    A.RandomResizedCrop(size=(640, 640), scale=(0.6, 1.0), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

test_transform = A.Compose([
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

model1 = Detector(
    backbone_model_name="efficientnet_b0",
    num_classes=4,
    unfreeze_last=2
).to(device)

optimizer1 = optim.AdamW(model1.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler1 = optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=20)
compute_loss1 = ComputeLoss(weight_bbox=5, weight_cls=1)

runner1 = Runner(
    model=model1,
    compute_loss=compute_loss1,
    optimizer=optimizer1,
    train_dataloader=train_dataloader,
    assign_target_method=assign_target,
    device=device,
    scheduler=scheduler1,
    assign_target_kwargs={"pos_th": 0.6, "neg_th": 0.4},
    val_dataloader=test_dataloader,
    val_every=2
)

model2 = Detector(
    backbone_model_name="efficientnet_b0",
    num_classes=4,
    unfreeze_last=2
).to(device)

optimizer2 = optim.AdamW(model2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=20)
compute_loss2 = ComputeLoss(weight_bbox=5, weight_cls=1)

runner2 = Runner(
    model=model2,
    compute_loss=compute_loss2,
    optimizer=optimizer2,
    train_dataloader=train_dataloader,
    assign_target_method=TAL_assigner,
    device=device,
    scheduler=scheduler2,
    assign_target_kwargs={"alpha": 1.0, "beta": 6.0, "topk": 13},
    val_dataloader=test_dataloader,
    val_every=2
)

print("Training with STANDARD assign_target...")
runner1.train(num_epochs=20, verbose=True)

print("Training with TAL_assigner...")
runner2.train(num_epochs=20, verbose=True)

print(f"Standard assigner final mAP: {runner1.val_metric[-1] if runner1.val_metric else 0:.4f}")
print(f"TAL assigner final mAP: {runner2.val_metric[-1] if runner2.val_metric else 0:.4f}")

# Визуализация сравнения
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
epochs = range(2, 21, 2)
plt.plot(epochs, runner1.val_metric, 'b-o', label='Standard Assigner')
plt.plot(epochs, runner2.val_metric, 'r-s', label='TAL Assigner')
plt.xlabel('Epoch')
plt.ylabel('mAP')
plt.title('Comparison of Label Assignment Methods')
plt.legend()
plt.grid(True)
plt.show()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 263.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [23]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    self.model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = self.model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()


ModuleNotFoundError: No module named 'torchmetrics'